# OHLCV Basics

OHLCV data is the standard daily market data format used for price analysis, indicators, and backtesting.

Abbreviations used in this notebook:

- **OHLCV**: Open, High, Low, Close, Volume.
- **VWAP**: Volume-Weighted Average Price, average traded price weighted by volume.
- **CHF**: Swiss franc, the currency used in the examples.

## 1. Intuition

Each trading day can be summarized by five fields. Open is where trading starts, high and low show the intraday range, close is the final traded price, and volume measures trading activity.

OHLCV data is useful because it records both price and participation. A price move with high volume often carries different information than the same move with low volume.

## 2. Mathematics

**Daily return:**

$$
R_t = \frac{Close_t}{Close_{t-1}} - 1
$$

Where:

- $R_t$ = simple return at time $t$
- $Close_t$ = closing price at time $t$
- $Close_{t-1}$ = previous closing price
- $t$ = time period index

**Log return:**

$$
r_t = \ln\left(\frac{Close_t}{Close_{t-1}}\right)
$$

Where:

- $r_t$ = log return at time $t$
- $Close_t$ = closing price at time $t$
- $Close_{t-1}$ = previous closing price
- $t$ = time period index

**Intraday range:**

$$
Range_t = \frac{High_t - Low_t}{Close_t}
$$

Where:

- $Close_t$ = closing price at time $t$
- $High_t$ = intraday high price at time $t$
- $Low_t$ = intraday low price at time $t$
- $Range_t$ = intraday range as a percentage of close
- $t$ = time period index

**Volume-weighted average price:**

$$
VWAP = \frac{\sum_t Close_t \times Volume_t}{\sum_t Volume_t}
$$

Where:

- $Close_t$ = closing price at time $t$
- $VWAP$ = volume-weighted average price
- $Volume_t$ = traded volume at time $t$
- $t$ = time period index

## 3. Implementation

We use the project pipeline helper to generate reproducible synthetic OHLCV data. This keeps the notebook executable without live market-data access.

In [ ]:
import importlib.util
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

project_root = next(path for path in [Path.cwd(), *Path.cwd().parents] if (path / "GUIDELINES.md").exists())
helper_path = project_root / "02_market_data" / "data_pipeline" / "fetch_data.py"
spec = importlib.util.spec_from_file_location("market_data_fetch_data", helper_path)
fetch_data = importlib.util.module_from_spec(spec)
spec.loader.exec_module(fetch_data)

add_return_columns = fetch_data.add_return_columns
generate_synthetic_ohlcv = fetch_data.generate_synthetic_ohlcv
save_ohlcv = fetch_data.save_ohlcv

plt.style.use("seaborn-v0_8-whitegrid")

ohlcv = generate_synthetic_ohlcv(ticker="NESN.SW", periods=180, seed=10)
ohlcv = add_return_columns(ohlcv)
ohlcv["intraday_range"] = (ohlcv["high"] - ohlcv["low"]) / ohlcv["close"]
ohlcv["vwap_running"] = ohlcv["traded_value"].cumsum() / ohlcv["volume"].cumsum()

save_ohlcv(ohlcv[["date", "ticker", "open", "high", "low", "close", "volume"]], project_root / "data" / "raw" / "synthetic_ohlcv.csv")
ohlcv.head()

In [ ]:
summary = pd.Series({
    "rows": len(ohlcv),
    "start_date": ohlcv["date"].min(),
    "end_date": ohlcv["date"].max(),
    "total_return": ohlcv["close"].iloc[-1] / ohlcv["close"].iloc[0] - 1,
    "average_daily_return": ohlcv["simple_return"].mean(),
    "daily_volatility": ohlcv["simple_return"].std(),
    "annualized_volatility": ohlcv["simple_return"].std() * np.sqrt(252),
    "average_volume": ohlcv["volume"].mean(),
    "sample_vwap": ohlcv["traded_value"].sum() / ohlcv["volume"].sum(),
})

summary.to_frame("value")

## 4. Visualization

The first check for OHLCV data is whether prices, ranges, volume, and returns look plausible.

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(11, 7), sharex=True, gridspec_kw={"height_ratios": [2, 1]})

axes[0].plot(ohlcv["date"], ohlcv["close"], color="#2f6f8f", label="Close")
axes[0].fill_between(ohlcv["date"], ohlcv["low"], ohlcv["high"], color="#2f6f8f", alpha=0.18, label="Daily range")
axes[0].plot(ohlcv["date"], ohlcv["vwap_running"], color="#9a6b2f", linestyle="--", label="Running VWAP")
axes[0].set_title("OHLCV Price Fields")
axes[0].set_ylabel("CHF per share")
axes[0].legend(loc="upper left")

axes[1].bar(ohlcv["date"], ohlcv["volume"], color="#9a6b2f", alpha=0.8)
axes[1].set_title("Volume")
axes[1].set_ylabel("Shares")
axes[1].set_xlabel("Date")

plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

axes[0].hist(ohlcv["simple_return"].dropna(), bins=24, color="#2f6f8f", edgecolor="white")
axes[0].axvline(0, color="black", linewidth=1)
axes[0].set_title("Daily Return Distribution")
axes[0].set_xlabel("Daily return")
axes[0].xaxis.set_major_formatter(lambda x, pos: f"{x:.1%}")

axes[1].plot(ohlcv["date"], ohlcv["intraday_range"], color="#9a6b2f")
axes[1].set_title("Intraday Range")
axes[1].set_xlabel("Date")
axes[1].set_ylabel("Range / close")
axes[1].yaxis.set_major_formatter(lambda x, pos: f"{x:.1%}")

plt.tight_layout()
plt.show()

## 5. Application

OHLCV data is the input layer for market data systems. It feeds technical indicators, return calculations, volatility estimates, portfolio backtests, and execution analysis.

Before using OHLCV data, always check missing dates, duplicated rows, adjusted versus unadjusted prices, outliers, and volume quality.

In [ ]:
quality_checks = pd.Series({
    "duplicate_date_ticker_rows": ohlcv.duplicated(["date", "ticker"]).sum(),
    "missing_close_values": ohlcv["close"].isna().sum(),
    "non_positive_volume_rows": (ohlcv["volume"] <= 0).sum(),
    "high_below_low_rows": (ohlcv["high"] < ohlcv["low"]).sum(),
    "close_outside_high_low_rows": ((ohlcv["close"] > ohlcv["high"]) | (ohlcv["close"] < ohlcv["low"])).sum(),
})

quality_checks.to_frame("count")

## 6. Reflection

- OHLCV is the raw grammar of market data.
- Returns are usually more useful than raw price levels for risk analysis.
- Volume helps interpret price moves and liquidity.
- Data quality checks are not optional; bad price data creates bad signals.

Questions to answer after running the notebook:

1. Why is volume useful alongside price?
2. Why do we calculate returns from close prices?
3. What data-quality problem would most damage a backtest?
4. When would VWAP be useful in real trading?